# Byte Pair Encoding (BPE)

Iteratively merge the most frequent adjacent token pair into a new token, growing the vocabulary from characters up to subwords.

## 1. Text & Character Inventory

In [60]:
text  = 'like liker love lovely hug hugs hugging hearts'
chars = sorted(set(text))

for c in chars:
    print(f'"{c}" — {text.count(c)} occurrences')

" " — 7 occurrences
"a" — 1 occurrences
"e" — 5 occurrences
"g" — 5 occurrences
"h" — 4 occurrences
"i" — 3 occurrences
"k" — 2 occurrences
"l" — 5 occurrences
"n" — 1 occurrences
"o" — 2 occurrences
"r" — 2 occurrences
"s" — 2 occurrences
"t" — 1 occurrences
"u" — 3 occurrences
"v" — 2 occurrences
"y" — 1 occurrences


In [61]:
vocab = list(chars)  # copy so merges don't mutate chars
vocab

[' ',
 'a',
 'e',
 'g',
 'h',
 'i',
 'k',
 'l',
 'n',
 'o',
 'r',
 's',
 't',
 'u',
 'v',
 'y']

## 2. BPE Functions

In [62]:
def get_pair_stats(tokens):
    pair_counts = {}
    for i in range(len(tokens) - 1):
        pair = tokens[i] + tokens[i+1]
        pair_counts[pair] = pair_counts.get(pair, 0) + 1
    return pair_counts

In [63]:
def update_vocab(pair_counts, vocab):
    new_token = max(pair_counts, key=pair_counts.get)
    vocab.append(new_token)
    return vocab, new_token

In [64]:
def merge_tokens(tokens, new_token):
    merged = []
    i = 0
    while i < len(tokens) - 1:
        if tokens[i] + tokens[i+1] == new_token:
            merged.append(new_token)
            i += 2
        else:
            merged.append(tokens[i])
            i += 1
    if i < len(tokens):
        merged.append(tokens[i])
    return merged

## 3. Run BPE

### a) Fixed number of merges

In [65]:
tokens = list(text)   # character-level tokenization
vocab  = list(chars)
n_merges = 10

for i in range(n_merges):
    pairs = get_pair_stats(tokens)
    vocab, new_token = update_vocab(pairs, vocab)
    tokens = merge_tokens(tokens, new_token)
    print(f'[{i+1:2d}] merged {new_token!r:10} → {len(tokens):3d} tokens, vocab size {len(vocab)}')

print(f'\nFinal tokens:\n{tokens}')
print(f'\nFinal vocab:\n{vocab}')

[ 1] merged ' h'       →  42 tokens, vocab size 17
[ 2] merged ' l'       →  39 tokens, vocab size 18
[ 3] merged ' hu'      →  36 tokens, vocab size 19
[ 4] merged ' hug'     →  33 tokens, vocab size 20
[ 5] merged 'ik'       →  31 tokens, vocab size 21
[ 6] merged 'ike'      →  29 tokens, vocab size 22
[ 7] merged ' lo'      →  27 tokens, vocab size 23
[ 8] merged ' lov'     →  25 tokens, vocab size 24
[ 9] merged ' love'    →  23 tokens, vocab size 25
[10] merged 'like'     →  22 tokens, vocab size 26

Final tokens:
['like', ' l', 'ike', 'r', ' love', ' love', 'l', 'y', ' hug', ' hug', 's', ' hug', 'g', 'i', 'n', 'g', ' h', 'e', 'a', 'r', 't', 's']

Final vocab:
[' ', 'a', 'e', 'g', 'h', 'i', 'k', 'l', 'n', 'o', 'r', 's', 't', 'u', 'v', 'y', ' h', ' l', ' hu', ' hug', 'ik', 'ike', ' lo', ' lov', ' love', 'like']


### b) Run until target vocab size

In [66]:
max_vocab_size = 25
tokens = list(text)
vocab  = list(chars)

while len(vocab) < max_vocab_size:
    pairs = get_pair_stats(tokens)
    vocab, new_token = update_vocab(pairs, vocab)
    tokens = merge_tokens(tokens, new_token)
    print(f'[{len(vocab):2d}] merged {new_token!r:10} → {len(tokens):3d} tokens, vocab size {len(vocab)}')

print(f'\nFinal tokens:\n{tokens}')
print(f'\nFinal vocab ({len(vocab)} tokens):\n{vocab}')

[17] merged ' h'       →  42 tokens, vocab size 17
[18] merged ' l'       →  39 tokens, vocab size 18
[19] merged ' hu'      →  36 tokens, vocab size 19
[20] merged ' hug'     →  33 tokens, vocab size 20
[21] merged 'ik'       →  31 tokens, vocab size 21
[22] merged 'ike'      →  29 tokens, vocab size 22
[23] merged ' lo'      →  27 tokens, vocab size 23
[24] merged ' lov'     →  25 tokens, vocab size 24
[25] merged ' love'    →  23 tokens, vocab size 25

Final tokens:
['l', 'ike', ' l', 'ike', 'r', ' love', ' love', 'l', 'y', ' hug', ' hug', 's', ' hug', 'g', 'i', 'n', 'g', ' h', 'e', 'a', 'r', 't', 's']

Final vocab (25 tokens):
[' ', 'a', 'e', 'g', 'h', 'i', 'k', 'l', 'n', 'o', 'r', 's', 't', 'u', 'v', 'y', ' h', ' l', ' hu', ' hug', 'ik', 'ike', ' lo', ' lov', ' love']
